# FALSETTO — Stage-1 training on a real GPU (Colab)

Trains the **Stage-1 segment detector** (frozen **MERT** features → **AudioCAT** head) and reports
real **Accuracy / Precision / Recall / F1 / AUC / Specificity** plus a **confusion matrix** on a
held-out test split.

**Runtime → Change runtime type → GPU (T4 is fine).**

### Why there is no faithful-benchmark mode

Every benchmark in these papers publishes only its **AI half**. Nobody owns AI output, so the
generated audio is redistributable; the **real** half is copyrighted commercial music, so all of it
ships as **YouTube ids instead of audio**:

- **FakeMusicCaps** real class = **MusicCaps**, which Google publishes as a CSV of YouTube ids. The
  Zenodo zip is 27,605 clips = 5 generators x 5,521 captions, with no reals. Its filenames *are*
  YouTube ids.
- **SONICS** ships `fake_songs/*.zip` plus a `real_songs.csv` carrying a `youtube_id` column.

So reproducing a published number means scraping tens of thousands of clips off YouTube, against a
list with well-known link rot. That yields a partial set whose score isn't comparable to the paper's
anyway. This notebook doesn't pretend otherwise.

### The three modes

- **`fmc_open`** *(default)* — the paper's **actual generated audio** (FakeMusicCaps, all 5 TTM
  models) against **open-licensed real music**. Full scale and free. This is an honest **proxy**, not
  the paper's benchmark: the AI half is theirs, the real half is not. Report it as a proxy.
- **`proxy`** — tiny; generates its own AI half with MusicGen. Minutes, weak metrics.
- **`fakemusiccaps`** — the faithful benchmark, if *you* have supplied the MusicCaps real audio.
  Point `DATA_ROOT` at a `real/` + per-model tree and skip to step 4.

## 1. Setup — clone, install, check GPU

In [ ]:
!nvidia-smi -L || echo "No GPU — set Runtime → Change runtime type → GPU"
!git clone -q https://github.com/arnavmahadev/Falsetto.git
%cd Falsetto
# Core install (MERT runs without nnAudio; it just skips the optional CQT branch).
!pip install -q . datasets
print("\nInstalled. If Colab warns about torch, it's fine — the preinstalled CUDA torch is kept.")

### 1b. Persist to Drive (strongly recommended)

Colab reclaims the VM when the session drops (a closed laptop is enough), taking the checkpoint
and both downloads with it. This mounts Drive and points `checkpoints/` at it, so `best.pt`
survives. The prepared clips get cached there too: they are ~1.6 GB against 23 GB of source
archives, so a second session skips the downloads entirely and starts training in about a minute.

In [ ]:
USE_DRIVE = True                                   # set False to keep everything on the VM
DRIVE_DIR = "/content/drive/MyDrive/falsetto"

if USE_DRIVE:
    import os

    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
    # `ckpt_dir` is relative to the cwd, so pointing the whole checkpoints/ tree at
    # Drive is enough to make every best.pt outlive the VM. No config change needed.
    !rm -rf /content/Falsetto/checkpoints
    !ln -sfn "{DRIVE_DIR}/checkpoints" /content/Falsetto/checkpoints
    print("checkpoints -> ", os.path.realpath("/content/Falsetto/checkpoints"))
else:
    print("not using Drive: a dropped session will lose the checkpoint and both downloads")

## 2. Choose the data mode + sizes

In [ ]:
# "fmc_open"      : paper's generated audio vs open real music. Honest proxy, full scale.
# "proxy"         : tiny MusicGen-generated AI half. Fast, weak.
# "fakemusiccaps" : faithful, but only if you have the MusicCaps real audio yourself.
DATA_MODE   = "fmc_open"

# Per class, so 2500 => 5,000 clips (~14 h of audio). Stage 1 runs MERT over every
# clip every epoch (frozen, but `augment` rules out caching), so this drives runtime:
# ~1-1.5 h end to end on a free T4. Early stopping usually ends it well before the
# config's 25 epochs. Raise it if your session is stable; the ceiling is 5,521 (the
# captions each generator rendered) and ~24,900 real clips.
N_PER_CLASS = 2500

# Both classes are pinned here. Non-negotiable: see the parity guard in 3c.
TARGET_SR   = 16000

DATA_ROOT = {"fmc_open": "data/raw/fmc_open", "proxy": "data/raw/proxy"}.get(DATA_MODE, "data/raw/fakemusiccaps")
MANIFEST  = {"fmc_open": "data/manifests/fmc_open.csv"}.get(DATA_MODE, "data/manifests/fakemusiccaps_small.csv")
CONFIG    = {"fmc_open": "configs/stage1_mert_fmc_open.yaml"}.get(DATA_MODE, "configs/stage1_mert_fakemusiccaps_small.yaml")

import os; os.makedirs("data/manifests", exist_ok=True)
print(f"mode={DATA_MODE} | {N_PER_CLASS}/class | sr={TARGET_SR} | config={CONFIG}")

## 3. Build the dataset

### 3a0. Reuse the clips a previous session prepared

The expensive part is turning 23 GB of archives into 5,000 matched 16 kHz clips. Those clips are
only ~1.6 GB, so they are cached on Drive and restored here. If this finds them, 3a and 3b are
skipped and you go straight to training.

In [ ]:
DATA_READY = False
if DATA_MODE == "fmc_open" and USE_DRIVE:
    import tarfile
    from pathlib import Path

    blob = Path(DRIVE_DIR) / "fmc_open_clips.tar"
    if blob.exists():
        print(f"restoring prepared clips from {blob} ({blob.stat().st_size/1e9:.2f} GB) ...")
        with tarfile.open(blob) as t:
            t.extractall("/content/Falsetto")
        n = len(list(Path(DATA_ROOT).rglob("*.wav")))
        DATA_READY = n > 100
        print(f"restored {n} clips -> skipping the 23 GB of downloads" if DATA_READY
              else f"only {n} clips restored; will rebuild from source")
    else:
        print("no cached clips on Drive yet; 3a/3b will build them and 3d will save them")

### 3a. The AI half — the paper's generated clips (Zenodo, open)

12.9 GB: 5 TTM models x 5,521 MusicCaps captions, 16 kHz mono. Downloads once and resumes if the
connection drops. Only `N_PER_CLASS // 5` captions per generator are extracted, and the *same*
captions are taken from each — the papers render every caption with all five, so grouping those
five variants under one `track_id` is what keeps the train/test split leak-free.

In [ ]:
if DATA_MODE == "fmc_open" and not DATA_READY:
    import time
    import zipfile
    from pathlib import Path

    import requests

    URL = "https://zenodo.org/api/records/15063698/files/FakeMusicCaps.zip/content"
    zp = Path("data/FakeMusicCaps.zip")
    zp.parent.mkdir(parents=True, exist_ok=True)

    # Fail fast and honestly if Zenodo itself is down. It returns a 503 HTML page,
    # and no amount of retrying fixes an outage; better to say so than to spin.
    head = requests.head(URL, allow_redirects=True, timeout=60)
    if head.status_code >= 500:
        raise SystemExit(
            f"Zenodo is returning HTTP {head.status_code} (their outage, not your connection "
            f"and not this notebook). Any partial download in data/ is kept, so just re-run "
            f"this cell once https://zenodo.org loads in a browser again."
        )
    head.raise_for_status()
    total = int(head.headers["Content-Length"])

    # Resume rather than restart: a 12.9 GB transfer will drop, and a truncated file
    # must never look complete to the next run.
    delay = 5
    for attempt in range(1, 9):
        done = zp.stat().st_size if zp.exists() else 0
        if done >= total:
            break
        print(f"[{attempt}] downloading from {done/1e9:.2f} / {total/1e9:.2f} GB")
        try:
            r = requests.get(URL, headers={"Range": f"bytes={done}-"}, stream=True, timeout=180)
            if r.status_code >= 500:
                raise SystemExit(
                    f"Zenodo went down mid-transfer (HTTP {r.status_code}). Your {done/1e9:.2f} GB "
                    f"is kept; re-run this cell when zenodo.org is back and it resumes from there."
                )
            r.raise_for_status()                      # 416 etc: a real bug, surface it
            with open(zp, "ab") as fh:
                for chunk in r.iter_content(8 << 20):
                    fh.write(chunk)
                    done += len(chunk)
                    print(f"\r    {done/1e9:6.2f} / {total/1e9:.2f} GB", end="")
            print()
        except SystemExit:
            raise
        except Exception as e:
            # Compare against the size before this attempt, not `done`, which only moves
            # when bytes actually land. The old check compared the file to itself and so
            # never fired, turning one outage into 20 hammers with no backoff.
            grew = (zp.stat().st_size if zp.exists() else 0) > done
            print(f"\n    dropped: {type(e).__name__}: {str(e)[:90]}")
            if not grew:
                print(f"    no progress; waiting {delay}s before retrying")
                time.sleep(delay)
                delay = min(delay * 2, 120)          # back off, do not hammer

    got = zp.stat().st_size if zp.exists() else 0
    if got != total:
        raise SystemExit(
            f"stopped at {got/1e9:.2f} / {total/1e9:.2f} GB after 8 attempts. The bytes so far "
            f"are kept: re-run this cell to carry on from here."
        )
    print(f"zip complete: {total/1e9:.2f} GB")

    per_model = max(1, N_PER_CLASS // 5)
    with zipfile.ZipFile(zp) as z:
        # __MACOSX/ holds an AppleDouble stub per clip ("._id.wav"); they end in .wav
        # and would otherwise scan as audio and be labelled by the dir they shadow.
        wavs = [n for n in z.namelist() if n.lower().endswith(".wav") and not n.startswith("__MACOSX")]
        by_model = {}
        for n in sorted(wavs):
            by_model.setdefault(n.split("/")[0], []).append(n)
        print("generators:", {m: len(v) for m, v in by_model.items()})

        keep = {Path(n).stem for n in by_model[sorted(by_model)[0]][:per_model]}
        n_out = 0
        for model, names in by_model.items():
            out = Path(DATA_ROOT) / model
            out.mkdir(parents=True, exist_ok=True)
            for n in names:
                if Path(n).stem in keep:
                    (out / Path(n).name).write_bytes(z.read(n))
                    n_out += 1
        print(f"extracted {n_out} generated clips ({len(keep)} captions x {len(by_model)} models) -> {DATA_ROOT}")

### 3b. The real half — open-licensed music, resampled to 16 kHz

**Why the resample matters more than anything else here.** The generated clips are 16 kHz, so they
carry nothing above 8 kHz. This source is 44.1 kHz stereo. Pair them as-is and the 8-12 kHz octave is
empty for every AI clip and full for every real one — the head scores ~1.0 by learning *the spectral
cutoff*, having learned nothing whatsoever about music. Holding both classes at 16 kHz mono removes
that shortcut. This is the same reason the paper's own real class is 16 kHz.

In [ ]:
if DATA_MODE == "fmc_open" and not DATA_READY:
    import io
    import time
    from pathlib import Path

    import librosa
    import soundfile as sf
    from datasets import Audio, load_dataset

    real_dir = Path(DATA_ROOT) / "real"
    real_dir.mkdir(parents=True, exist_ok=True)

    def to_mono_16k(entry):
        """Match the generated clips exactly: mono, TARGET_SR, first 10 s.

        Only the first 10 s is decoded and resampled. Doing the whole 30 s clip and
        then slicing was ~13x slower for a byte-identical result.
        """
        raw = entry.get("bytes")
        src = io.BytesIO(raw) if raw else entry["path"]
        with sf.SoundFile(src) as f:
            sr = f.samplerate
            y = f.read(frames=sr * 10, dtype="float32", always_2d=True)
        y = y.mean(axis=1)                                    # stereo -> mono
        if sr != TARGET_SR:
            y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)  # lowpasses at 8 kHz
        return y[: TARGET_SR * 10]

    # Full (resumable) download, not streaming -> survives the RemoteProtocolError on
    # big parquet files. decode=False + soundfile sidesteps datasets' audio-decode churn.
    saved, t0 = 0, time.time()
    for repo in ["lewtun/music_genres", "lewtun/music_genres_small"]:
        if saved >= N_PER_CLASS:
            break
        try:
            ds = load_dataset(repo, split="train").cast_column("audio", Audio(decode=False))
        except Exception as e:
            print(f"  {repo} unavailable ({type(e).__name__}); trying next")
            continue
        for ex in ds:
            try:
                y = to_mono_16k(ex["audio"])
            except Exception:
                continue
            if len(y) < TARGET_SR * 9.5:                      # keep the length uniform
                continue
            sf.write(real_dir / f"real_{saved:05d}.wav", y, TARGET_SR)
            saved += 1
            # Progress, because this loop is otherwise silent for minutes and looks hung
            if saved % 100 == 0:
                rate = saved / max(time.time() - t0, 1e-6)
                print(f"\r  {saved}/{N_PER_CLASS} real clips  ({rate:.1f}/s)", end="")
            if saved >= N_PER_CLASS:
                break
        print(f"\r  {repo}: {saved}/{N_PER_CLASS} real clips" + " " * 20)

    assert saved >= 100, "could not fetch enough real music; re-run (the drop is transient)"
    print(f"saved {saved} real clips @ {TARGET_SR} Hz mono -> {real_dir}")

### 3c. Bandwidth-parity guard

The check that decides whether the final number means anything. If the classes disagree on sample
rate or channel count, stop: the score would be measuring the format, not the music.

In [ ]:
if DATA_MODE == "fmc_open":
    import random
    from pathlib import Path

    import soundfile as sf

    rng = random.Random(0)
    rates, chans = set(), set()
    print(f"{'class dir':22s} {'clips':>7s}  {'sample rates':>14s}  channels")
    for d in sorted(p for p in Path(DATA_ROOT).iterdir() if p.is_dir()):
        files = list(d.glob("*.wav"))
        if not files:
            continue
        info = [sf.info(str(f)) for f in rng.sample(files, min(25, len(files)))]
        srs = {i.samplerate for i in info}
        chs = {i.channels for i in info}
        rates |= srs
        chans |= chs
        print(f"  {d.name:20s} {len(files):7d}  {str(sorted(srs)):>14s}  {sorted(chs)}")

    assert rates == {TARGET_SR}, (
        f"sample rates differ across classes: {sorted(rates)}. The model would learn the "
        f"cutoff, not the music. Re-run 3b."
    )
    assert chans == {1}, f"channel counts differ across classes: {sorted(chans)}"
    print(f"\nOK — every class is {TARGET_SR} Hz mono. No bandwidth shortcut available.")

### 3d. Cache the prepared clips on Drive

Runs once, right after the parity guard passes. It is the difference between a 35-minute restart
and a 1-minute one.

In [ ]:
if DATA_MODE == "fmc_open" and USE_DRIVE and not DATA_READY:
    import tarfile
    from pathlib import Path

    blob = Path(DRIVE_DIR) / "fmc_open_clips.tar"
    # Tarred, not copied: Drive is very slow with thousands of small files.
    # Uncompressed on purpose, since 16-bit PCM barely compresses and gzip would
    # cost minutes of CPU for a few percent.
    print(f"caching prepared clips -> {blob} (~1.6 GB, once) ...")
    tmp = blob.with_suffix(".tar.part")
    with tarfile.open(tmp, "w") as t:
        t.add(DATA_ROOT, arcname=DATA_ROOT)
    tmp.replace(blob)          # atomic: a half-written tar must never look complete
    print(f"cached {blob.stat().st_size/1e9:.2f} GB. Next session restores in ~1 min.")

### 3d. Real music (proxy mode only)

In [ ]:
if DATA_MODE == "proxy":
    import soundfile as sf, numpy as np, tempfile, os
    from pathlib import Path
    from datasets import load_dataset, Audio

    real_dir = Path(DATA_ROOT) / "real"; real_dir.mkdir(parents=True, exist_ok=True)
    # Full (resumable) download, not streaming -> survives the flaky-connection
    # RemoteProtocolError on big parquet files. decode=False + soundfile sidesteps
    # datasets' audio-decode version churn. Script-based sets (marsyas/gtzan) are
    # unsupported on modern `datasets`, so they're omitted.
    SOURCES = ["lewtun/music_genres_small", "lewtun/music_genres"]

    def to_wave(entry):
        raw, path = entry.get("bytes"), entry.get("path")
        if raw:
            path = tempfile.mktemp(suffix=os.path.splitext(path or "")[1] or ".wav")
            with open(path, "wb") as fh:
                fh.write(raw)
        try:
            y, sr = sf.read(path, dtype="float32")
        except Exception:
            import librosa
            y, sr = librosa.load(path, sr=None, mono=False)
            y = np.asarray(y, "float32")
            y = y.T if y.ndim > 1 else y
        return y, int(sr)

    saved = 0
    for repo in SOURCES:
        try:
            ds = load_dataset(repo, split="train").cast_column("audio", Audio(decode=False))
            for ex in ds:
                y, sr = to_wave(ex["audio"])
                if y.ndim > 1:
                    y = y.mean(axis=1)               # -> mono
                y = y[: sr * 10]                     # first 10 s
                if len(y) < sr * 3:                  # skip too-short
                    continue
                sf.write(real_dir / f"real_{saved:04d}.wav", y, sr)
                saved += 1
                if saved >= N_PER_CLASS:
                    break
            print(f"  {repo}: have {saved}/{N_PER_CLASS} real clips")
            if saved >= N_PER_CLASS:
                break
        except Exception as e:
            print(f"  {repo} failed ({type(e).__name__}: {e}); trying next source...")

    assert saved >= 10, ("Could not fetch real music from any source. Re-run this cell (the drop is "
                         "transient), or set DATA_MODE='fakemusiccaps' with your own data.")
    print(f"saved {saved} real clips -> {real_dir}")

### 3e. AI music (proxy mode only) — generate with MusicGen

In [ ]:
if DATA_MODE == "proxy":
    import torch, soundfile as sf
    from pathlib import Path
    from transformers import AutoProcessor, MusicgenForConditionalGeneration

    fake_dir = Path(DATA_ROOT) / "musicgen"; fake_dir.mkdir(parents=True, exist_ok=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    proc = AutoProcessor.from_pretrained("facebook/musicgen-small")
    mg = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(dev)
    sr = mg.config.audio_encoder.sampling_rate

    PROMPTS = [
        "an upbeat pop song with drums and synths", "a calm lo-fi hip hop beat",
        "an energetic rock track with electric guitar", "a smooth jazz piano piece",
        "an epic orchestral cinematic score", "a funky disco groove with bass",
        "a dreamy ambient electronic soundscape", "a latin salsa dance rhythm",
        "a country folk song with acoustic guitar", "a hard techno club track",
        "a soulful r&b ballad", "a reggae tune with offbeat guitar",
        "a classical string quartet", "an 80s synthwave retro track",
        "a bright indie pop melody",
    ]
    # Batch on GPU so a few hundred clips finish in minutes, not an hour.
    BATCH = 8 if dev == "cuda" else 1
    i = 0
    while i < N_PER_CLASS:
        prompts = [PROMPTS[(i + k) % len(PROMPTS)] for k in range(min(BATCH, N_PER_CLASS - i))]
        inputs = proc(text=prompts, padding=True, return_tensors="pt").to(dev)
        with torch.no_grad():
            audio = mg.generate(**inputs, do_sample=True, max_new_tokens=500)  # ~10 s each
        for b in range(audio.shape[0]):
            wav = audio[b, 0].float().cpu().numpy()
            sf.write(fake_dir / f"gen_{i:04d}_musicgen.wav", wav, sr)
            i += 1
        print(f"  generated {i}/{N_PER_CLASS}")
    print(f"saved {i} AI clips -> {fake_dir}")

> **`fakemusiccaps` mode:** set `DATA_MODE="fakemusiccaps"` and point `DATA_ROOT` at a tree laid out
> as `real/` + `MusicGen_medium/` `musicldm/` `audioldm2/` `stable_audio_open/` `mustango/` (see
> `docs/DATASETS.md`). You have to supply `real/` yourself. Skip 3a-3d and go to step 4.

## 4. Build a balanced, leak-free manifest (8:1:1 split)

In [ ]:
!python scripts/build_manifest.py --root "$DATA_ROOT" --out "$MANIFEST" \
    --max-per-class $N_PER_CLASS --seed 42

## 5. Train Stage-1 (frozen MERT → AudioCAT)
The extractor is frozen; only the AudioCAT head trains. First step downloads MERT (~350 MB, cached).


In [ ]:
import os

import yaml

# Auto-resume: if a previous session left a checkpoint on Drive, continue from it
# rather than paying for those epochs twice.
CKPT = f"checkpoints/{yaml.safe_load(open(CONFIG))['name']}/best.pt"
resume = f"--resume {CKPT}" if os.path.exists(CKPT) else ""
print(f"resuming from {CKPT}" if resume else "starting fresh")

!falsetto train --config "$CONFIG" --stage 1 $resume

## 6. Evaluate — metrics table + confusion matrix

In [ ]:
import matplotlib.pyplot as plt, numpy as np, json
from falsetto.config import load_config
from falsetto.eval.table_stage1 import evaluate_checkpoint, results_to_markdown

cfg = load_config(CONFIG)
ckpt = f"checkpoints/{cfg.name}/best.pt"
res = evaluate_checkpoint(cfg, ckpt, split="test")

print(results_to_markdown({"MERT + AudioCAT": res}, title="Stage-1 (test split)"))

e = res.extra
cm = np.array([[e["tn"], e["fp"]], [e["fn"], e["tp"]]])
fig, ax = plt.subplots(figsize=(4.2, 3.8))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], ["pred real", "pred AI"])
ax.set_yticks([0, 1], ["true real", "true AI"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=15)
ax.set_title(f"Stage-1 confusion (test) — acc={res.accuracy:.2f}  f1={res.f1:.2f}  auc={res.auc:.2f}")
fig.tight_layout()
import os; os.makedirs("results", exist_ok=True)
fig.savefig("results/stage1_confusion.png", dpi=150)
plt.show()
print("saved -> results/stage1_confusion.png  (download it for the results page)")

# ---- Copy this block back to fill in FALSETTO's row on the results page ----
print("\n===== COPY THIS BACK =====")
print(json.dumps({
    "task": {
        "fmc_open": "PROXY: open real music vs FakeMusicCaps 5 TTM generators (16 kHz matched)",
        "proxy": "PROXY: real music vs MusicGen",
    }.get(DATA_MODE, "FakeMusicCaps Stage-1 (faithful)"),
    "faithful_benchmark": DATA_MODE == "fakemusiccaps",
    "n_per_class": N_PER_CLASS,
    "accuracy": round(res.accuracy, 4), "precision": round(res.precision, 4),
    "recall": round(res.recall, 4), "f1": round(res.f1, 4),
    "auc": round(res.auc, 4), "specificity": round(res.specificity, 4),
    "confusion": {"tn": e["tn"], "fp": e["fp"], "fn": e["fn"], "tp": e["tp"]},
}, indent=2))

## 7. Notes

- **Proxy numbers look high** because real-vs-MusicGen is easier than real-vs-many-generators. Report
  them as *"the Stage-1 pipeline trains and separates real from AI-generated audio"* — not as the
  paper's benchmark. For the faithful result, run mode `fakemusiccaps`.
- **Scale up** by raising `N_PER_CLASS` and the `epochs` in
  `configs/stage1_mert_fakemusiccaps_small.yaml`.
- **Save the checkpoint** (`checkpoints/stage1_mert_fakemusiccaps_small/best.pt`) to Drive if you want
  to reuse it — Colab wipes local files when the session ends.
